## Build Tract-Year Panel for Difference-in-Differences
                                                                                                                                                                                            
Constructs the panel dataset needed for the DiD analysis following Mamkhezri, Sun & Yang (2026):                                                                                             
                                                                                                                                                                                            
1. Aggregate pnode LMPs to census tract level by year                                                                                                                                        
2. Create inefficient pricing = congestion + marginal loss
3. Merge with data center first entry years                                                                                                                                                  
4. Add treatment indicator (DC_it = 1 if year >= first entry year)
5. Restrict to matched sample tracts from notebook 07                                                                                                                                        
                                                                                                                                                                                            
**Output:** `data/processed/for_analysis/va_did_panel.csv` — with tract-year LMP averages, treatment indicator, event   
time, and matching weights                                                                                                                                                                   
                


In [41]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

lmp_processed_dir = DATA_DIR / "processed" / "preprocessing" / "lmp_data"   
matching_dir = DATA_DIR / "processed" / "preprocessing" / "pre_analysis"     
analysis_dir = DATA_DIR / "processed" / "for_analysis"

In [42]:
# Load data
lmp = pd.read_csv(lmp_processed_dir / "va_lmp_yearly_avg_load_geo.csv", dtype={"census_geoid": str})
dc = pd.read_excel(analysis_dir / "datacenters_matched_tracts.xlsx",
                   dtype={"census_tract_geoid": str})
matched = pd.read_csv(matching_dir / "va_matched_tracts.csv", dtype={"GEOID": str})

print("LMP shape:", lmp.shape)
print("Data centers shape:", dc.shape)
print("Matched tracts shape:", matched.shape)
print("\nLMP years:", sorted(lmp["year"].unique()))
print("LMP columns:", lmp.columns.tolist())

LMP shape: (8886, 13)
Data centers shape: (292, 43)
Matched tracts shape: (60, 14)

LMP years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
LMP columns: ['year', 'pnode_id', 'pnode_name', 'type', 'avg_total_lmp', 'avg_energy', 'avg_congestion', 'avg_loss', 'hours', 'latitude', 'longitude', 'census_geoid', 'census_name']


In [43]:
# Step 1: Aggregate pnode LMPs to census tract level by year
# Note: va_lmp_yearly_avg_geo.csv is already filtered to LOAD type only
tract_lmp = lmp.groupby(["census_geoid", "census_name", "year"]).agg(
    avg_total_lmp=("avg_total_lmp", "mean"),
    avg_energy=("avg_energy", "mean"),
    avg_congestion=("avg_congestion", "mean"),
    avg_loss=("avg_loss", "mean"),
    n_pnodes=("pnode_id", "count")
).reset_index()

# Step 2: Create inefficient pricing = congestion + marginal loss
tract_lmp["inefficient_pricing"] = tract_lmp["avg_congestion"] + tract_lmp["avg_loss"]

print(f"Tract-year observations: {len(tract_lmp):,}")
print(f"Unique tracts: {tract_lmp['census_geoid'].nunique()}")
tract_lmp.head(11)

Tract-year observations: 3,792
Unique tracts: 355


,census_geoid,census_name,year,avg_total_lmp,avg_energy,avg_congestion,avg_loss,n_pnodes,inefficient_pricing
0,51001090100,901.0,2015,62.212420,33.339051,25.601115,3.272253,2,28.873369
1,51001090100,901.0,2016,47.004851,27.524868,17.533644,1.946339,2,19.479983
2,51001090100,901.0,2017,46.050955,29.388410,14.560212,2.102333,2,16.662546
3,51001090100,901.0,2018,57.927592,35.693402,19.211818,3.022603,2,22.234421
4,51001090100,901.0,2019,28.691388,25.979946,1.159640,1.552088,2,2.711728
5,51001090100,901.0,2020,23.415566,20.625790,1.792975,0.997053,2,2.790027
6,51001090100,901.0,2021,48.380848,38.125223,8.166065,2.089809,2,10.255874
7,51001090100,901.0,2022,81.380116,73.388705,2.856699,5.135034,2,7.991733
8,51001090100,901.0,2023,40.291725,29.612192,8.881726,1.797913,2,10.679639
9,51001090100,901.0,2024,41.630460,31.229977,8.166217,2.234514,2,10.400731


In [44]:
# Step 3: Get first data center entry year per tract                                                                                                                                   
first_entry = dc.groupby("census_tract_geoid")["source_year_found"].min().reset_index()
first_entry.columns = ["census_geoid", "first_entry_year"]    

# Step 4: Merge LMP panel with first entry years and add treatment indicator
panel = tract_lmp.merge(first_entry, on="census_geoid", how="left")

# DC_it = 1 if year >= first_entry_year, 0 otherwise
panel["treated"] = (
    panel["first_entry_year"].notna() &
    (panel["year"] >= panel["first_entry_year"])
).astype(int)

# Event time: years relative to first entry (negative = pre-treatment)
panel["event_time"] = panel["year"] - panel["first_entry_year"]

print(f"Panel rows: {len(panel):,}")
print(f"\nTreated observations: {panel['treated'].sum()}")
print(f"Untreated observations: {(panel['treated'] == 0).sum()}")

Panel rows: 3,792

Treated observations: 146
Untreated observations: 3646


In [45]:
# Step 5: Restrict to matched sample tracts
matched_geoids = set(matched["GEOID"])
panel_matched = panel[panel["census_geoid"].isin(matched_geoids)].copy()

# Merge in weights from matched sample
weights = matched[["GEOID", "weight", "group"]].drop_duplicates(subset="GEOID")
weights = weights.rename(columns={"GEOID": "census_geoid"})
panel_matched = panel_matched.merge(weights, on="census_geoid", how="left")

print(f"Panel rows after restricting to matched sample: {len(panel_matched):,}")
print(f"Unique tracts: {panel_matched['census_geoid'].nunique()}")
print(f"Treated tracts: {panel_matched[panel_matched['group'] == 'treated']['census_geoid'].nunique()}")
print(f"Control tracts: {panel_matched[panel_matched['group'] == 'control']['census_geoid'].nunique()}")
panel_matched.head()

Panel rows after restricting to matched sample: 637
Unique tracts: 60
Treated tracts: 11
Control tracts: 49


,census_geoid,census_name,year,avg_total_lmp,avg_energy,avg_congestion,avg_loss,n_pnodes,inefficient_pricing,first_entry_year,treated,event_time,weight,group
0,51003010300,103.0,2015,36.288607,33.339051,3.199473,-0.249918,3,2.949555,NaN,0,NaN,0.5,control
1,51003010300,103.0,2016,29.559131,27.524868,2.374874,-0.340612,3,2.034263,NaN,0,NaN,0.5,control
2,51003010300,103.0,2017,31.109871,29.388410,1.889254,-0.167793,3,1.721461,NaN,0,NaN,0.5,control
3,51003010300,103.0,2018,38.520143,35.693402,3.099526,-0.272803,3,2.826723,NaN,0,NaN,0.5,control
4,51003010300,103.0,2019,26.997804,25.979946,1.225924,-0.207923,3,1.018002,NaN,0,NaN,0.5,control


In [46]:
# Check treated tracts all have data centers                                                                                                                                           
dc_geoids = set(dc["census_tract_geoid"].dropna().unique())                                                                                                                            
                                                                                                                                                                                        
treated_tracts = set(panel_matched[panel_matched["group"] == "treated"]["census_geoid"].unique())                                                                                      
control_tracts = set(panel_matched[panel_matched["group"] == "control"]["census_geoid"].unique())                                                                                      
                                                                                                                                                                                        
treated_with_dc = treated_tracts & dc_geoids                                                                                                                                           
treated_without_dc = treated_tracts - dc_geoids                                                                                                                                        
control_with_dc = control_tracts & dc_geoids                                                                                                                                           
control_without_dc = control_tracts - dc_geoids
                                                                                                                                                                                        
print(f"Treated tracts WITH data centers: {len(treated_with_dc)} / {len(treated_tracts)}")                                                                                             
print(f"Treated tracts WITHOUT data centers: {len(treated_without_dc)}")
print(f"\nControl tracts WITHOUT data centers: {len(control_without_dc)} / {len(control_tracts)}")                                                                                     
print(f"Control tracts WITH data centers: {len(control_with_dc)}")                                                                                                                     

if control_with_dc:                                                                                                                                                                    
    print("\nControl tracts that have data centers:")
    print(control_with_dc)  

Treated tracts WITH data centers: 11 / 11
Treated tracts WITHOUT data centers: 0

Control tracts WITHOUT data centers: 49 / 49
Control tracts WITH data centers: 0


In [47]:
# Summary statistics by treatment group
print("Mean inefficient pricing by group:")
print(panel_matched.groupby("group")["inefficient_pricing"].describe().round(3))

Mean inefficient pricing by group:
         count   mean    std    min    25%    50%    75%     max
group                                                           
control  534.0  4.968  5.874 -4.250  2.054  3.567  5.977  45.501
treated  103.0  6.502  9.709 -4.288  2.071  3.550  6.386  57.621


In [49]:
# Diagnose tracts without full 2015-2025 coverage                                                                                                                     
all_years = set(range(2015, 2026))                                                                                                                                    
                                                                                                                                                                    
coverage = (                                                                                                                                                          
    check.groupby(["census_geoid", "group", "first_entry_year"])["year"]                                                                                              
    .apply(set)                                                                                                                                                       
    .reset_index()                                                                                                                                                    
)                                                                                                                                                                     
coverage["missing_years"] = coverage["year"].apply(lambda y: sorted(all_years - y))                                                                                 
coverage["n_years"] = coverage["year"].apply(len)                                                                                                                     
coverage["n_missing"] = coverage["missing_years"].apply(len)                                                                                                        
                                                                                                                                                                    
incomplete = coverage[coverage["n_missing"] > 0].sort_values(["group", "n_missing"], ascending=[False, False])                                                        
                                                                                                                                                                    
if len(incomplete) == 0:                                                                                                                                              
    print("All tracts have full 2015-2025 coverage.")                                                                                                               
else:                                                                                                                                                                 
    print(f"Tracts with incomplete coverage: {len(incomplete)}")
    print()                                                                                                                                                           
    for _, row in incomplete.iterrows():                                                                                                                            
        print(f"  {row['census_geoid']} ({row['group']}, first entry: {row['first_entry_year']})")
        print(f"    Years present: {row['n_years']}/11 | Missing: {row['missing_years']}")                                                                            
                                                                                                                                                                    
# Flag treated tracts with no pre-treatment data                                                                                                                      
print("\nTreated tracts with no pre-treatment LMP observations:")                                                                                                     
treated_rows = check[check["group"] == "treated"]                                                                                                                     
no_pretreat = (
    treated_rows[treated_rows["treated"] == 0]                                                                                                                        
    .groupby("census_geoid")["year"].count()                                                                                                                          
)
tracts_no_pretreat = set(treated_rows["census_geoid"].unique()) - set(no_pretreat.index)                                                                              
if tracts_no_pretreat:                                                                                                                                              
    print(f"  {tracts_no_pretreat}  ← these cannot be used in DiD")                                                                                                   
else:                                                                                                                                                                 
    print("  None — all treated tracts have at least one pre-treatment year.")  

Tracts with incomplete coverage: 5

  51153901303 (treated, first entry: 2021.0)
    Years present: 2/11 | Missing: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  51107611601 (treated, first entry: 2024.0)
    Years present: 7/11 | Missing: [2015, 2016, 2017, 2018]
  51107611801 (treated, first entry: 2022.0)
    Years present: 9/11 | Missing: [2015, 2016]
  51107611804 (treated, first entry: 2019.0)
    Years present: 9/11 | Missing: [2015, 2016]
  51107611004 (treated, first entry: 2022.0)
    Years present: 10/11 | Missing: [2015]

Treated tracts with no pre-treatment LMP observations:
  {'51153901303'}  ← these cannot be used in DiD


We remove tract 51153901303 from the panel, as it has no pre-treatment years in the data we have. This is because there is no lmp data (at least in our dataset) for that tract for those years (ie. new pnodes were created in that tract in later years).

In [50]:
# Drop tract 51153901303 — only 2 years of data (2024, 2025), both post-treatment.                                                                                    
# First entry year is 2021, so it has zero pre-treatment observations and cannot                                                                                      
# contribute to DiD estimation.                                                                                                                                       
before = len(panel_matched)                                                                                                                                           
panel_matched = panel_matched[panel_matched["census_geoid"] != "51153901303"].copy()                                                                                  
print(f"Dropped tract 51153901303: {before - len(panel_matched)} rows removed")                                                                                       
print(f"Remaining: {len(panel_matched):,} rows | "                                                                                                                    
    f"Treated tracts: {panel_matched[panel_matched['group'] == 'treated']['census_geoid'].nunique()} | "                                                            
    f"Control tracts: {panel_matched[panel_matched['group'] == 'control']['census_geoid'].nunique()}")  

Dropped tract 51153901303: 2 rows removed
Remaining: 635 rows | Treated tracts: 10 | Control tracts: 49


In [51]:
# Save panel
panel_matched.to_csv(analysis_dir / "va_did_panel.csv", index=False)
print(f"Saved {len(panel_matched):,} rows to va_did_panel.csv")

Saved 635 rows to va_did_panel.csv


In [52]:
# Verify saved panel                                                                                                                                                                           
check = pd.read_csv(analysis_dir / "va_did_panel.csv", dtype={"census_geoid": str})                                                                                                            
treated_tracts = check[check["group"] == "treated"]["census_geoid"].nunique()                                                                                                                  
control_tracts = check[check["group"] == "control"]["census_geoid"].nunique()                                                                                                                  
print(f"Treated tracts: {treated_tracts}")                                                                                                                                                     
print(f"Control tracts: {control_tracts}")                                                                                                                                                     
print(f"Total rows: {len(check):,}")    

Treated tracts: 10
Control tracts: 49
Total rows: 635
